# EGFR Wild-type vs Mutant: Docking & Resistance Analysis

**Clinical story:** EGFR TKI resistance evolution in NSCLC

```
Wild-type ──(1st gen TKI)──> L858R activation ──(resistance)──> T790M gatekeeper
                                                                      │
                                                         (3rd gen TKI: Osimertinib)
                                                                      │
                                                                T790M/C797S
```

## Structures

| # | Label | PDB | Mutation | Drug | Generation |
|---|-------|-----|----------|------|------------|
| 1 | WT + Erlotinib | 1M17 | Wild-type | Erlotinib | 1st gen |
| 2 | WT + Gefitinib | 2ITY | Wild-type | Gefitinib | 1st gen |
| 3 | WT + Lapatinib | 1XKK | Wild-type (inactive) | Lapatinib | Dual EGFR/HER2 |
| 4 | WT + Afatinib | 4G5J | Wild-type | Afatinib | 2nd gen |
| 5 | L858R + Gefitinib | 2ITZ | L858R | Gefitinib | 1st gen |
| 6 | T790M + Neratinib | 2JIT | T790M | HKI-272 | 2nd gen |
| 7 | T790M/L858R + WZ4002 | 3IKA | T790M/L858R | WZ4002 | 3rd gen-like |

## Cross-docking Drugs

| Drug | Generation | Mechanism | Target |
|------|-----------|-----------|--------|
| Erlotinib | 1st | Reversible | WT, L858R |
| Gefitinib | 1st | Reversible | WT, L858R |
| Afatinib | 2nd | Irreversible | WT, L858R, T790M (weak) |
| Osimertinib | 3rd | Irreversible | T790M selective |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/ddtut1/blob/main/EGFR_WT_Mutant_Docking_Colab.ipynb)

---
## 1. Environment Setup

In [ ]:
#@title 1-1. Install Packages {display-mode: "form"}
!pip install -q rdkit-pypi
!pip install -q meeko vina
!pip install -q openbabel-wheel
!pip install -q pdbfixer openmm
!pip install -q pymol-open-source
!pip install -q py3Dmol
!pip install -q MDAnalysis
!pip install -q prolif
!pip install -q seaborn scipy scikit-learn
print('\n=== Packages installed ===')

In [ ]:
#@title 1-2. Install Binary Tools {display-mode: "form"}
import os, stat, urllib.request, tarfile, shutil

BIN_DIR = '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)

def _chmod_x(path):
    os.chmod(path, os.stat(path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# smina
if not os.path.exists(f'{BIN_DIR}/smina'):
    print('Downloading smina...')
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download',
        f'{BIN_DIR}/smina')
    _chmod_x(f'{BIN_DIR}/smina')

# ADFRsuite
ADFR_DIR = f'{BIN_DIR}/ADFRsuite'
if not os.path.exists(ADFR_DIR):
    print('Downloading ADFRsuite...')
    urllib.request.urlretrieve('https://ccsb.scripps.edu/adfr/download/1038/', '/tmp/adfr.tar.gz')
    with tarfile.open('/tmp/adfr.tar.gz') as tar:
        tar.extractall('/tmp/')
    d = [x for x in os.listdir('/tmp/') if x.startswith('ADFRsuite')][0]
    try:
        os.chdir(f'/tmp/{d}')
        !echo 'Y' | ./install.sh -d {ADFR_DIR} -c 0 > /dev/null 2>&1
    except Exception as e:
        print(f'ADFRsuite install warning: {e}')
    finally:
        os.chdir('/content')

for tool in ['prepare_receptor', 'prepare_ligand']:
    wrapper = f'{BIN_DIR}/{tool}'
    src = f'{ADFR_DIR}/bin/{tool}'
    if os.path.exists(src) and not os.path.exists(wrapper):
        with open(wrapper, 'w') as f:
            f.write(f'#!/bin/bash\n{src} "$@"\n')
        _chmod_x(wrapper)

os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print('=== Tools ready ===')

In [ ]:
#@title 1-3. Imports {display-mode: "form"}
from pymol import cmd
import py3Dmol
from vina import Vina
from openbabel import pybel

from rdkit import Chem
from rdkit.Chem import AllChem, Draw, PandasTools, DataStructs

from pdbfixer import PDBFixer
from openmm.app import PDBFile

import MDAnalysis as mda
from MDAnalysis.coordinates import PDB
import prolif as plf
from prolif.plotting.network import LigNetwork

import os, glob, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from collections import defaultdict

warnings.filterwarnings('ignore')
BIN_DIR = '/content/bin'
WORK_ROOT = '/content/egfr_docking'
os.makedirs(WORK_ROOT, exist_ok=True)
print('Libraries imported.')

---
## 2. EGFR Structures & Reference Drugs

In [ ]:
#@title 2-1. Define EGFR Complexes {display-mode: "form"}

EGFR_COMPLEXES = [
    # ---- Wild-type ----
    {'label': 'WT_Erlotinib',    'pdb': '1M17', 'chain': 'A',
     'mutation': 'Wild-type', 'conformation': 'Active (DFG-in)',
     'drug': 'Erlotinib',  'generation': '1st gen',
     'notes': 'First EGFR-TKI co-crystal, Tarceva'},

    {'label': 'WT_Gefitinib',    'pdb': '2ITY', 'chain': 'A',
     'mutation': 'Wild-type', 'conformation': 'Active (DFG-in)',
     'drug': 'Gefitinib',  'generation': '1st gen',
     'notes': 'Iressa complex'},

    {'label': 'WT_Lapatinib',    'pdb': '1XKK', 'chain': 'A',
     'mutation': 'Wild-type', 'conformation': 'Inactive (DFG-out)',
     'drug': 'Lapatinib',  'generation': 'Dual EGFR/HER2',
     'notes': 'Type II inhibitor, captures inactive conformation'},

    {'label': 'WT_Afatinib',     'pdb': '4G5J', 'chain': 'A',
     'mutation': 'Wild-type', 'conformation': 'Active (DFG-in)',
     'drug': 'Afatinib',   'generation': '2nd gen',
     'notes': 'Irreversible pan-ErbB inhibitor, Gilotrif'},

    # ---- Activating mutation ----
    {'label': 'L858R_Gefitinib', 'pdb': '2ITZ', 'chain': 'A',
     'mutation': 'L858R', 'conformation': 'Active',
     'drug': 'Gefitinib',  'generation': '1st gen',
     'notes': 'Most common activating point mutation in exon 21'},

    # ---- Resistance mutations ----
    {'label': 'T790M_Neratinib', 'pdb': '2JIT', 'chain': 'A',
     'mutation': 'T790M', 'conformation': 'Active',
     'drug': 'HKI-272 (Neratinib)', 'generation': '2nd gen',
     'notes': 'Gatekeeper mutation, confers 1st-gen resistance'},

    {'label': 'DM_WZ4002',       'pdb': '3IKA', 'chain': 'A',
     'mutation': 'T790M/L858R', 'conformation': 'Active',
     'drug': 'WZ4002',     'generation': '3rd gen-like',
     'notes': 'Double mutant, precursor to osimertinib design'},
]

egfr_df = pd.DataFrame(EGFR_COMPLEXES)
egfr_df.index += 1
egfr_df[['label', 'pdb', 'mutation', 'drug', 'generation', 'conformation']]

In [ ]:
#@title 2-2. Define Reference Drug Panel (SMILES) {display-mode: "form"}

REFERENCE_DRUGS = {
    'Erlotinib': {
        'smiles': 'C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1',
        'generation': '1st gen', 'mechanism': 'Reversible',
        'target': 'WT, L858R sensitive; T790M resistant'},
    'Gefitinib': {
        'smiles': 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',
        'generation': '1st gen', 'mechanism': 'Reversible',
        'target': 'WT, L858R sensitive; T790M resistant'},
    'Afatinib': {
        'smiles': 'CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OC1CCOC1',
        'generation': '2nd gen', 'mechanism': 'Irreversible (C797)',
        'target': 'Broad: WT, L858R, T790M (weak)'},
    'Osimertinib': {
        'smiles': 'C=CC(=O)Nc1cc(OC)c(Nc2nccc(-c3cn(C)c4ccccc34)n2)cc1N(C)CCN(C)C',
        'generation': '3rd gen', 'mechanism': 'Irreversible (C797)',
        'target': 'T790M selective, spares WT'},
}

# Visualize drugs
drug_mols = []
drug_labels = []
for name, info in REFERENCE_DRUGS.items():
    mol = Chem.MolFromSmiles(info['smiles'])
    if mol:
        drug_mols.append(mol)
        drug_labels.append(f"{name}\n({info['generation']})")

if drug_mols:
    img = Draw.MolsToGridImage(drug_mols, legends=drug_labels,
                                molsPerRow=4, subImgSize=(350, 250))
    display(img)

---
## 3. Pipeline Functions

In [ ]:
#@title 3-1. Utility Functions {display-mode: "form"}

def getbox(selection='sele', extending=6.0):
    """Compute Vina docking box from PyMOL selection."""
    ([minX, minY, minZ], [maxX, maxY, maxZ]) = cmd.get_extent(selection)
    pad = float(extending)
    minX -= pad; minY -= pad; minZ -= pad
    maxX += pad; maxY += pad; maxZ += pad
    return (
        {'center_x': (maxX+minX)/2, 'center_y': (maxY+minY)/2, 'center_z': (maxZ+minZ)/2},
        {'size_x': maxX-minX, 'size_y': maxY-minY, 'size_z': maxZ-minZ}
    )


def fix_protein(filename, output, addHs_pH=7.4, try_renumber=False):
    """Fix missing residues/atoms, add H with PDBFixer."""
    fixer = PDBFixer(filename=filename)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(True)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(addHs_pH)
    with open(output, 'w') as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f)
    if try_renumber:
        try:
            orig = mda.Universe(filename)
            fixed = mda.Universe(output)
            for i, res in enumerate(fixed.residues):
                res.resid = orig.residues[i].resid
            w = PDB.PDBWriter(filename=output); w.write(fixed); w.close()
        except Exception:
            pass


def pdbqt_to_sdf(pdbqt_file, output):
    """Convert Vina PDBQT output → SDF with Pose/Score metadata."""
    out = pybel.Outputfile(filename=output, format='sdf', overwrite=True)
    for mol in pybel.readfile(filename=pdbqt_file, format='pdbqt'):
        mol.data['Pose'] = mol.data.get('MODEL', '?')
        mol.data['Score'] = mol.data.get('REMARK', '').split()[2] if 'REMARK' in mol.data else '0'
        for k in ['MODEL', 'REMARK', 'TORSDO']:
            mol.data.pop(k, None)
        out.write(mol)
    out.close()


def smiles_to_sdf(smiles, name, output_path):
    """SMILES → 3D-embedded SDF file."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp('_Name', name)
    mol.SetProp('ID', name)
    w = Chem.SDWriter(output_path)
    w.write(mol)
    w.close()
    return output_path


print('Utility functions ready.')

In [ ]:
#@title 3-2. EGFRDockingPipeline Class {display-mode: "form"}

class EGFRPipeline:
    """
    Docking pipeline for a single EGFR structure.

    Usage
    -----
    >>> p = EGFRPipeline('WT_Erlotinib', '1M17', mutation='Wild-type')
    >>> p.prepare()
    >>> p.dock_native()                          # re-dock co-crystal ligand
    >>> p.dock_drug('Osimertinib', smiles)       # cross-dock a reference drug
    >>> p.get_scores('native')                   # scores DataFrame
    >>> p.get_prolif('native')                   # ProLIF fingerprint
    >>> p.view_3d('native', pose_index=0)        # 3D viewer
    """

    def __init__(self, label, pdb_code, chain='A', mutation='WT', work_dir=None):
        self.label = label
        self.pdb = pdb_code.lower()
        self.chain = chain
        self.mutation = mutation
        self.work_dir = work_dir or f'{WORK_ROOT}/{label}'
        os.makedirs(self.work_dir, exist_ok=True)

        self.clean_pdb = None; self.lig_mol2 = None
        self.prot_H = None; self.lig_H = None
        self.rec_qt = None; self.lig_qt = None
        self.center = None; self.size = None
        self.results = {}   # label -> sdf_path
        self.prepared = False

    def _p(self, fn):
        return os.path.join(self.work_dir, fn)

    # ---- Preparation steps ----
    def prepare(self):
        """Full preparation: fetch → clean → fix → PDBQT."""
        try:
            # 1. Fetch & clean
            cmd.delete('all')
            self.clean_pdb = self._p(f'{self.pdb}_clean.pdb')
            self.lig_mol2 = self._p(f'{self.pdb}_lig.mol2')

            cmd.fetch(self.pdb, type='pdb', path=self.work_dir)
            cmd.remove(f'not chain {self.chain}')
            cmd.select('Prot', 'polymer.protein')
            cmd.select('Lig', 'organic')
            n_lig = cmd.count_atoms('Lig')

            if n_lig < 5:
                # Retry: keep all chains for ligand
                cmd.delete('all')
                cmd.fetch(self.pdb, type='pdb', path=self.work_dir)
                cmd.select('Prot', f'chain {self.chain} and polymer.protein')
                cmd.select('Lig', 'organic and not resn HOH+EDO+PEG+GOL+DMS+SO4+PO4+ACT')
                n_lig = cmd.count_atoms('Lig')

            cmd.save(self.clean_pdb, 'Prot')
            cmd.save(self.lig_mol2, 'Lig')
            cmd.delete('all')
            print(f'  [{self.label}] PDB {self.pdb}: ligand {n_lig} atoms')

            if n_lig < 5:
                print(f'  [{self.label}] No ligand found — skipping.')
                return False

            # 2. Add H to ligand
            self.lig_H = self._p(f'{self.pdb}_lig_H.mol2')
            mol = list(pybel.readfile(self.lig_mol2, 'mol2'))[0]
            mol.addh()
            out = pybel.Outputfile(filename=self.lig_H, format='mol2', overwrite=True)
            out.write(mol); out.close()

            # 3. Fix protein & compute docking box
            self.prot_H = self._p(f'{self.pdb}_clean_H.pdb')
            fix_protein(self.clean_pdb, self.prot_H, addHs_pH=7.2, try_renumber=True)
            cmd.delete('all')
            cmd.load(self.prot_H, 'prot')
            cmd.load(self.lig_H, 'lig')
            self.center, self.size = getbox('lig', extending=7.0)
            cmd.delete('all')
            print(f'  [{self.label}] Box center: ({self.center["center_x"]:.1f}, '
                  f'{self.center["center_y"]:.1f}, {self.center["center_z"]:.1f})')

            # 4. PDBQT
            self.rec_qt = self._p(f'{self.pdb}_clean_H.pdbqt')
            self.lig_qt = self._p(f'{self.pdb}_lig_H.pdbqt')
            os.system(f'{BIN_DIR}/prepare_receptor -r {self.prot_H} -o {self.rec_qt} > /dev/null 2>&1')
            os.system(f'{BIN_DIR}/prepare_ligand -l {self.lig_H} -o {self.lig_qt} > /dev/null 2>&1')
            self.prepared = os.path.exists(self.rec_qt) and os.path.exists(self.lig_qt)
            print(f'  [{self.label}] PDBQT: {"OK" if self.prepared else "FAILED"}')
            return self.prepared

        except Exception as e:
            print(f'  [{self.label}] ERROR: {e}')
            return False

    # ---- Docking ----
    def _run_smina(self, lig_file, out_sdf, exhaustiveness=64, n_poses=20):
        c, s = self.center, self.size
        os.system(
            f'{BIN_DIR}/smina -r {self.rec_qt} -l {lig_file} -o {out_sdf} '
            f'--center_x {c["center_x"]} --center_y {c["center_y"]} --center_z {c["center_z"]} '
            f'--size_x {s["size_x"]} --size_y {s["size_y"]} --size_z {s["size_z"]} '
            f'--exhaustiveness {exhaustiveness} --num_modes {n_poses} > /dev/null 2>&1')
        return os.path.exists(out_sdf) and os.path.getsize(out_sdf) > 0

    def dock_native(self, exhaustiveness=64, n_poses=20):
        """Re-dock co-crystallized ligand."""
        if not self.prepared: return None
        out = self._p(f'{self.pdb}_native_smina.sdf')
        if self._run_smina(self.lig_qt, out, exhaustiveness, n_poses):
            self.results['native'] = out
            print(f'  [{self.label}] Native docking done.')
            return out
        print(f'  [{self.label}] Native docking FAILED.')
        return None

    def dock_drug(self, drug_name, smiles, exhaustiveness=48, n_poses=10):
        """Dock a reference drug by SMILES."""
        if not self.prepared: return None
        sdf_in = self._p(f'{drug_name}_3d.sdf')
        if not os.path.exists(sdf_in):
            smiles_to_sdf(smiles, drug_name, sdf_in)
        out = self._p(f'{self.pdb}_{drug_name}_smina.sdf')
        if self._run_smina(sdf_in, out, exhaustiveness, n_poses):
            self.results[drug_name] = out
            return out
        return None

    # ---- Analysis ----
    def get_scores(self, label='native'):
        sdf = self.results.get(label)
        if not sdf or not os.path.exists(sdf): return pd.DataFrame()
        df = PandasTools.LoadSDF(sdf, smilesName='SMILES', molColName='Molecule',
                                 includeFingerprints=False, strictParsing=False)
        if 'Score' in df.columns:
            df = df.rename(columns={'Score': 'minimizedAffinity'})
        if 'minimizedAffinity' in df.columns:
            df['minimizedAffinity'] = pd.to_numeric(df['minimizedAffinity'], errors='coerce')
        df['structure'] = self.label
        df['mutation'] = self.mutation
        df['drug_label'] = label
        return df

    def get_prolif(self, label='native', pose_indices=None):
        sdf = self.results.get(label)
        if not sdf: return None, None, None
        prot_mol = Chem.MolFromPDBFile(self.prot_H, removeHs=False)
        if prot_mol is None:
            u = mda.Universe(self.prot_H, guess_bonds=True)
            prot_mol = plf.Molecule.from_mda(u)
        else:
            prot_mol = plf.Molecule(prot_mol)
        ligs = list(plf.sdf_supplier(sdf))
        if pose_indices:
            ligs = [ligs[i] for i in pose_indices if i < len(ligs)]
        fp = plf.Fingerprint()
        fp.run_from_iterable(ligs, prot_mol)
        return fp, fp.to_dataframe(return_atoms=True), ligs

    def view_3d(self, label='native', pose_index=0):
        sdf = self.results.get(label)
        if not sdf: return None
        v = py3Dmol.view(width=800, height=500)
        v.removeAllModels()
        v.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})
        v.addModel(open(self.prot_H).read(), 'pdb')
        v.getModel().setStyle({'cartoon': {'arrows': True, 'tubes': True, 'style': 'oval', 'color': 'white'}})
        v.addSurface(py3Dmol.VDW, {'opacity': 0.5, 'color': 'white'})
        if os.path.exists(self.lig_H):
            v.addModel(open(self.lig_H).read(), 'mol2')
            v.getModel().setStyle({}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.15}})
        poses = [m for m in Chem.SDMolSupplier(sdf, True) if m]
        if pose_index < len(poses):
            v.addModel(Chem.MolToMolBlock(poses[pose_index]), 'mol')
            v.getModel().setStyle({}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.25}})
        v.zoomTo()
        return v

    def __repr__(self):
        s = 'ready' if self.prepared else 'not prepared'
        return f'EGFRPipeline({self.label}, {self.pdb}, {self.mutation}, {s}, {len(self.results)} dockings)'


print('EGFRPipeline class ready.')

---
## 4. Batch Preparation

In [ ]:
#@title 4-1. Prepare All EGFR Structures {display-mode: "form"}
%%time

pipes = {}
for i, cx in enumerate(EGFR_COMPLEXES, 1):
    print(f'\n--- [{i}/{len(EGFR_COMPLEXES)}] {cx["label"]} ({cx["pdb"]}) ---')
    p = EGFRPipeline(cx['label'], cx['pdb'], cx['chain'], cx['mutation'])
    if p.prepare():
        pipes[cx['label']] = p

print(f'\n=== {len(pipes)}/{len(EGFR_COMPLEXES)} structures prepared ===')
for p in pipes.values():
    print(f'  {p}')

---
## 5. Native Re-docking

In [ ]:
#@title 5-1. Re-dock Co-crystallized Ligands {display-mode: "form"}
%%time

for label, p in pipes.items():
    p.dock_native(exhaustiveness=64, n_poses=20)

print(f'\n=== Native re-docking complete ===')

In [ ]:
#@title 5-2. Native Re-docking Scores {display-mode: "form"}

native_scores = []
for label, p in pipes.items():
    df = p.get_scores('native')
    if not df.empty:
        cx = next(c for c in EGFR_COMPLEXES if c['label'] == label)
        df['generation'] = cx['generation']
        df['conformation'] = cx['conformation']
        native_scores.append(df)

native_df = pd.concat(native_scores, ignore_index=True)

summary = native_df.groupby(['structure', 'mutation']).agg(
    best=('minimizedAffinity', 'min'),
    mean=('minimizedAffinity', 'mean'),
    n_poses=('minimizedAffinity', 'count')
).round(2).sort_values('best')
print('Native re-docking scores (kcal/mol):')
summary

---
## 6. Cross-docking: All Drugs × All Structures

Dock 4 generation TKIs into all 7 EGFR structures to reveal resistance patterns.

In [ ]:
#@title 6-1. Cross-dock Reference Drugs (~15-20 min) {display-mode: "form"}
%%time

xdock_scores = {}  # (drug, structure) -> best_score

for drug_name, drug_info in REFERENCE_DRUGS.items():
    print(f'\n--- Docking {drug_name} ({drug_info["generation"]}) ---')
    for label, p in pipes.items():
        sdf = p.dock_drug(drug_name, drug_info['smiles'], exhaustiveness=48, n_poses=10)
        if sdf:
            df = p.get_scores(drug_name)
            if not df.empty and 'minimizedAffinity' in df.columns:
                best = df['minimizedAffinity'].min()
                xdock_scores[(drug_name, label)] = best
                print(f'  {label:25s} → {best:.2f} kcal/mol')

print(f'\n=== Cross-docking done: {len(xdock_scores)} pairs ===')

In [ ]:
#@title 6-2. Cross-docking Heatmap (Drugs × Structures) {display-mode: "form"}

drug_names = list(REFERENCE_DRUGS.keys())
struct_labels = list(pipes.keys())

matrix = pd.DataFrame(np.nan, index=struct_labels, columns=drug_names)
for (drug, struct), score in xdock_scores.items():
    matrix.loc[struct, drug] = score

# Annotate mutation type on index
mut_map = {c['label']: c['mutation'] for c in EGFR_COMPLEXES}
row_labels = [f"{l} ({mut_map.get(l, '')})" for l in matrix.index]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(matrix.astype(float), annot=True, fmt='.1f', cmap='RdYlGn',
            center=-7.0, ax=ax, linewidths=0.8, yticklabels=row_labels,
            cbar_kws={'label': 'Binding Affinity (kcal/mol)'})
ax.set_xlabel('Drug (→ newer generation)')
ax.set_ylabel('EGFR Structure (mutation)')
ax.set_title('EGFR Cross-docking: Drug Generation × Mutation Status\n'
             '(green=strong binding, red=weak binding)')
plt.tight_layout()
plt.show()

In [ ]:
#@title 6-3. Drug Sensitivity by Mutation Type {display-mode: "form"}

# Aggregate scores by mutation
xdock_data = []
for (drug, struct), score in xdock_scores.items():
    mut = mut_map.get(struct, 'Unknown')
    gen = REFERENCE_DRUGS[drug]['generation']
    xdock_data.append({'drug': drug, 'structure': struct, 'mutation': mut,
                       'generation': gen, 'score': score})

xdock_df = pd.DataFrame(xdock_data)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By drug per mutation
mut_colors = {'Wild-type': '#3498DB', 'L858R': '#E67E22',
              'T790M': '#E74C3C', 'T790M/L858R': '#8E44AD',
              'Wild-type (inactive)': '#95A5A6'}

for mut in xdock_df['mutation'].unique():
    sub = xdock_df[xdock_df['mutation'] == mut]
    avg = sub.groupby('drug')['score'].mean()
    axes[0].plot(avg.index, avg.values, 'o-', label=mut,
                 color=mut_colors.get(mut, 'gray'), markersize=8, linewidth=2)
axes[0].set_ylabel('Best Binding Affinity (kcal/mol)')
axes[0].set_title('Drug Sensitivity by EGFR Mutation')
axes[0].legend()
axes[0].axhline(y=-7.0, color='gray', linestyle='--', alpha=0.5)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

# Grouped barplot
pivot = xdock_df.groupby(['mutation', 'drug'])['score'].mean().unstack()
pivot.plot(kind='bar', ax=axes[1], edgecolor='black', linewidth=0.3, width=0.7)
axes[1].set_ylabel('Best Binding Affinity (kcal/mol)')
axes[1].set_title('Cross-docking Scores: Mutation × Drug')
axes[1].legend(title='Drug', bbox_to_anchor=(1.02, 1))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
#@title 6-4. Resistance Analysis: WT vs T790M {display-mode: "form"}

# Compare 1st gen drug scores in WT vs T790M
print('=== Resistance Analysis ===')
print('\n1st gen TKIs: How much does T790M reduce binding?')
print('-' * 60)

for drug in ['Erlotinib', 'Gefitinib']:
    wt_scores = [s for (d, st), s in xdock_scores.items()
                 if d == drug and 'WT' in st and 'Lapatinib' not in st]
    t790m_scores = [s for (d, st), s in xdock_scores.items()
                    if d == drug and ('T790M' in mut_map.get(st, ''))]
    if wt_scores and t790m_scores:
        wt_avg = np.mean(wt_scores)
        t790m_avg = np.mean(t790m_scores)
        delta = t790m_avg - wt_avg
        print(f'  {drug:12s}: WT avg={wt_avg:.2f}, T790M avg={t790m_avg:.2f}, '
              f'\u0394={delta:+.2f} kcal/mol')

print('\n3rd gen TKI: Does Osimertinib selectively bind T790M?')
print('-' * 60)
for drug in ['Osimertinib']:
    for label, p in pipes.items():
        if (drug, label) in xdock_scores:
            mut = mut_map.get(label, '')
            score = xdock_scores[(drug, label)]
            marker = ' <<<' if 'T790M' in mut else ''
            print(f'  {label:25s} ({mut:15s}): {score:.2f} kcal/mol{marker}')

---
## 7. ProLIF Interaction Comparison

In [ ]:
#@title 7-1. ProLIF for Native Ligands {display-mode: "form"}

prolif_data = {}
for label, p in pipes.items():
    try:
        fp, df, ligs = p.get_prolif('native', pose_indices=[0])
        if fp is not None:
            prolif_data[label] = (fp, df, ligs)
            n_int = df.any().sum() if df is not None else 0
            print(f'  [{label}] {n_int} interactions')
    except Exception as e:
        print(f'  [{label}] Failed: {e}')

print(f'\nProLIF computed for {len(prolif_data)}/{len(pipes)} structures.')

In [ ]:
#@title 7-2. WT vs Mutant Interaction Comparison {display-mode: "form"}

INTERACTION_TYPES = ['Hydrophobic', 'HBDonor', 'HBAcceptor', 'PiStacking',
                     'PiCation', 'CationPi', 'Cationic', 'Anionic']

int_counts = {}
for label, (fp, df, _) in prolif_data.items():
    counts = {}
    if df is not None and not df.empty:
        for itype in INTERACTION_TYPES:
            cols = [c for c in df.columns if itype in str(c)]
            counts[itype] = df[cols].any().sum() if cols else 0
    int_counts[label] = counts

int_df = pd.DataFrame(int_counts).T.fillna(0).astype(int)

# Add mutation annotation
mut_annot = [mut_map.get(l, '') for l in int_df.index]

fig, ax = plt.subplots(figsize=(14, 6))
int_df.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='black', linewidth=0.3)
ax.set_ylabel('Number of Interacting Residues')
ax.set_title('Protein-Ligand Interactions: WT vs Mutant EGFR')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')

# Color x-labels by mutation
for i, (tick, mut) in enumerate(zip(ax.get_xticklabels(), mut_annot)):
    tick.set_color(mut_colors.get(mut, 'black'))
    tick.set_fontweight('bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#@title 7-3. LigNetwork for Selected Structure {display-mode: "form"}

VIEW_STRUCTURE = 'WT_Erlotinib'  #@param {type:"string"}

if VIEW_STRUCTURE in prolif_data:
    fp, df, ligs = prolif_data[VIEW_STRUCTURE]
    net = LigNetwork.from_ifp(df, ligs[0], kind='frame', frame=0, rotation=270)
    print(f'Interaction network: {VIEW_STRUCTURE}')
    net.display()
else:
    print(f'Available: {list(prolif_data.keys())}')

In [ ]:
#@title 7-4. ProLIF for Cross-docked Osimertinib {display-mode: "form"}

# Compare Osimertinib interactions in WT vs T790M
osi_prolif = {}
for label, p in pipes.items():
    if 'Osimertinib' in p.results:
        try:
            fp, df, ligs = p.get_prolif('Osimertinib', pose_indices=[0])
            if fp:
                osi_prolif[label] = (fp, df, ligs)
        except Exception:
            pass

if osi_prolif:
    osi_int = {}
    for label, (fp, df, _) in osi_prolif.items():
        counts = {}
        for itype in INTERACTION_TYPES:
            cols = [c for c in df.columns if itype in str(c)]
            counts[itype] = df[cols].any().sum() if cols else 0
        osi_int[f'{label}\n({mut_map.get(label, "")})'] = counts

    osi_df = pd.DataFrame(osi_int).T.fillna(0).astype(int)

    fig, ax = plt.subplots(figsize=(12, 5))
    osi_df.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='black', linewidth=0.3)
    ax.set_ylabel('Number of Interacting Residues')
    ax.set_title('Osimertinib (3rd gen) Interactions Across EGFR Variants')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No Osimertinib cross-docking data. Run section 6 first.')

---
## 7.5 Advanced Resistance Analysis

RMSD validation, ligand efficiency, consensus scoring, IFP-based PCA, shape similarity.

In [ ]:
#@title 7.5-1. RMSD Validation {display-mode: "form"}
from rdkit.Chem import rdMolAlign

rmsd_data = []
for label, p in pipes.items():
    sdf = p.results.get("native")
    if not sdf or not os.path.exists(p.lig_H): continue
    ref = Chem.MolFromMol2File(p.lig_H, removeHs=False)
    if ref is None:
        pmol = list(pybel.readfile(p.lig_H, "mol2"))[0]
        pmol.write("pdb", "/tmp/_ref.pdb", overwrite=True)
        ref = Chem.MolFromPDBFile("/tmp/_ref.pdb", removeHs=False)
    if ref is None: continue
    ref_noH = Chem.RemoveHs(ref)
    for i, mol in enumerate(Chem.SDMolSupplier(sdf, removeHs=False)):
        if mol is None or i >= 5: continue
        try:
            rmsd = rdMolAlign.GetBestRMS(ref_noH, Chem.RemoveHs(mol))
            rmsd_data.append({"structure": label, "mutation": p.mutation,
                              "pose": i, "RMSD": round(rmsd, 2)})
        except Exception: pass

if rmsd_data:
    rmsd_df = pd.DataFrame(rmsd_data)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=rmsd_df, x="structure", y="RMSD", palette="Set3", ax=ax)
    ax.axhline(y=2.0, color="red", linestyle="--", label="2.0 \u00c5 threshold")
    ax.set_ylabel("RMSD (\u00c5)"); ax.set_title("Re-docking RMSD: WT vs Mutant")
    ax.legend()
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()
    
    best_rmsd = rmsd_df.groupby("structure")["RMSD"].min()
    print("Best RMSD per structure:")
    for s, r in best_rmsd.items():
        status = "PASS" if r < 2.0 else "FAIL"
        print(f"  {s:25s}: {r:.2f} \u00c5 [{status}]")


In [ ]:
#@title 7.5-2. Ligand Efficiency by Mutation {display-mode: "form"}
from rdkit.Chem import Descriptors

le_rows = []
for (drug, struct), score in xdock_scores.items():
    # Get molecule from reference drugs
    if drug in REFERENCE_DRUGS:
        mol = Chem.MolFromSmiles(REFERENCE_DRUGS[drug]["smiles"])
        if mol:
            ha = mol.GetNumHeavyAtoms()
            mw = Descriptors.MolWt(mol)
            le_rows.append({"drug": drug, "structure": struct,
                            "mutation": mut_map.get(struct, "?"),
                            "score": score, "HA": ha,
                            "LE": round(-score/ha, 3) if ha > 0 else 0,
                            "BEI": round(-score/(mw/1000), 2) if mw > 0 else 0,
                            "generation": REFERENCE_DRUGS[drug]["generation"]})

if le_rows:
    le_df = pd.DataFrame(le_rows)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # LE by drug × mutation
    pivot_le = le_df.groupby(["mutation", "drug"])["LE"].mean().unstack()
    pivot_le.plot(kind="bar", ax=axes[0], edgecolor="black", linewidth=0.3)
    axes[0].axhline(y=0.3, color="green", linestyle="--")
    axes[0].set_ylabel("Ligand Efficiency"); axes[0].set_title("LE: Mutation × Drug")
    axes[0].legend(title="Drug", bbox_to_anchor=(1.02, 1))
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")
    
    # BEI comparison
    pivot_bei = le_df.groupby(["mutation", "drug"])["BEI"].mean().unstack()
    pivot_bei.plot(kind="bar", ax=axes[1], edgecolor="black", linewidth=0.3)
    axes[1].set_ylabel("Binding Efficiency Index"); axes[1].set_title("BEI: Mutation × Drug")
    axes[1].legend(title="Drug", bbox_to_anchor=(1.02, 1))
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha="right")
    plt.tight_layout(); plt.show()


In [ ]:
#@title 7.5-3. Consensus Scoring: WT vs T790M {display-mode: "form"}

scoring_fns = ["vina", "vinardo", "ad4_scoring"]
cons_data = []

for label, p in pipes.items():
    if not p.prepared: continue
    for sf in scoring_fns:
        out = p._p(f"{p.pdb}_{sf}_rescore.sdf")
        p._run_smina(p.lig_qt, out, exhaustiveness=32, n_poses=5)
        if os.path.exists(out) and os.path.getsize(out) > 0:
            df = PandasTools.LoadSDF(out, strictParsing=False)
            if "minimizedAffinity" in df.columns:
                best = pd.to_numeric(df["minimizedAffinity"], errors="coerce").min()
                cons_data.append({"structure": label, "mutation": p.mutation,
                                  "SF": sf, "best_score": best})

if cons_data:
    cons_df = pd.DataFrame(cons_data)
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot = cons_df.pivot_table(index="structure", columns="SF", values="best_score")
    pivot.plot(kind="bar", ax=ax, edgecolor="black", linewidth=0.3)
    ax.set_ylabel("Best Score (kcal/mol)")
    ax.set_title("Consensus Scoring: Multiple Scoring Functions")
    ax.legend(title="Scoring Function")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()
    
    # Rank correlation
    print("\nSpearman correlation between scoring functions:")
    rank_pivot = pivot.rank()
    print(rank_pivot.corr("spearman").round(2))


In [ ]:
#@title 7.5-4. IFP-based PCA: WT vs Mutant Clustering {display-mode: "form"}
from sklearn.decomposition import PCA

# Collect IFP vectors from cross-docked Osimertinib
osi_ifps = {}
for label, p in pipes.items():
    if "Osimertinib" not in p.results: continue
    try:
        fp, df, ligs = p.get_prolif("Osimertinib", pose_indices=[0])
        if df is not None and not df.empty:
            osi_ifps[label] = df.iloc[0].to_dict()
    except Exception: pass

# Also add native ligand IFPs
native_ifps = {}
for label, (fp, df, _) in prolif_data.items():
    if df is not None and not df.empty:
        native_ifps[f"{label}_native"] = df.iloc[0].to_dict()

all_ifps = {**{f"{k}_osi": v for k, v in osi_ifps.items()}, **native_ifps}

if len(all_ifps) >= 3:
    all_keys = set()
    for v in all_ifps.values(): all_keys.update(v.keys())
    mat = pd.DataFrame({n: {k: int(bool(v.get(k, False))) for k in all_keys}
                        for n, v in all_ifps.items()}).T
    
    pca = PCA(n_components=2)
    coords = pca.fit_transform(mat.values)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    for (name, (x, y)) in zip(mat.index, coords):
        color = "#E74C3C" if "T790M" in name else "#3498DB" if "WT" in name else "#2ECC71"
        marker = "s" if "_osi" in name else "o"
        ax.scatter(x, y, color=color, s=120, marker=marker, edgecolors="black", zorder=5)
        ax.annotate(name.replace("_native","(N)").replace("_osi","(O)"),
                    (x+0.02, y+0.02), fontsize=8)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
    ax.set_title("PCA of Interaction Fingerprints\n(N)=native ligand, (O)=Osimertinib")
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend = [Patch(color="#3498DB", label="WT"), Patch(color="#E74C3C", label="T790M+"),
              Patch(color="#2ECC71", label="L858R"),
              Line2D([0],[0], marker="o", color="gray", label="Native", linestyle=""),
              Line2D([0],[0], marker="s", color="gray", label="Osimertinib", linestyle="")]
    ax.legend(handles=legend, loc="best")
    plt.tight_layout(); plt.show()
else:
    print(f"Need >=3 IFP vectors, have {len(all_ifps)}")


In [ ]:
#@title 7.5-5. Pharmacophore Feature Comparison {display-mode: "form"}
from rdkit.Chem import rdMolDescriptors

# Extract pharmacophore features for each drug
pharma_data = []
for drug_name, drug_info in REFERENCE_DRUGS.items():
    mol = Chem.MolFromSmiles(drug_info["smiles"])
    if mol is None: continue
    mol = Chem.AddHs(mol)
    pharma_data.append({
        "drug": drug_name,
        "generation": drug_info["generation"],
        "MW": round(Descriptors.MolWt(mol), 1),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "RotBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "TPSA": round(Descriptors.TPSA(mol), 1),
        "LogP": round(Descriptors.MolLogP(mol), 2),
        "Rings": rdMolDescriptors.CalcNumRings(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
    })

pharma_df = pd.DataFrame(pharma_data).set_index("drug")
print("Drug Physicochemical Properties:")
display(pharma_df)

# Radar/spider plot
props = ["MW", "HBA", "HBD", "RotBonds", "TPSA", "LogP", "Rings"]
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
norm = pd.DataFrame(scaler.fit_transform(pharma_df[props]),
                    index=pharma_df.index, columns=props)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2*np.pi, len(props), endpoint=False).tolist()
angles += angles[:1]
gen_colors = {"1st gen": "#3498DB", "2nd gen": "#E67E22", "3rd gen": "#E74C3C",
              "Dual EGFR/HER2": "#9B59B6"}
for drug in norm.index:
    vals = norm.loc[drug].values.tolist() + [norm.loc[drug].values[0]]
    gen = pharma_df.loc[drug, "generation"]
    ax.plot(angles, vals, "o-", label=f"{drug} ({gen})",
            color=gen_colors.get(gen, "gray"), linewidth=2)
    ax.fill(angles, vals, alpha=0.1, color=gen_colors.get(gen, "gray"))
ax.set_xticks(angles[:-1]); ax.set_xticklabels(props)
ax.set_title("Drug Pharmacophore Property Comparison", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.show()


---
## 8. Structural Comparison

In [ ]:
#@title 8-1. Binding Site Comparison: WT vs Mutants {display-mode: "form"}

site_data = []
for label, p in pipes.items():
    if not p.prepared:
        continue
    cmd.delete('all')
    cmd.load(p.prot_H, 'prot')
    cmd.load(p.lig_H, 'lig')
    cmd.select('site', 'prot within 5.0 of lig')

    # Key EGFR residues
    n_total = cmd.count_atoms('site and name CA')
    n_charged = cmd.count_atoms('site and name CA and (resn ASP+GLU+LYS+ARG+HIS)')
    n_aromatic = cmd.count_atoms('site and name CA and (resn PHE+TYR+TRP+HIS)')
    n_hydrophobic = cmd.count_atoms('site and name CA and (resn ALA+VAL+LEU+ILE+MET+PRO)')
    n_polar = cmd.count_atoms('site and name CA and (resn SER+THR+ASN+GLN+CYS)')
    n_gly = cmd.count_atoms('site and name CA and resn GLY')

    # Check gatekeeper position (T790/M790)
    has_met790 = cmd.count_atoms('resn MET and resi 790 and name CA') > 0
    # Check C797 (irreversible TKI target)
    has_cys797 = cmd.count_atoms('resn CYS and resi 797 and name CA') > 0

    site_data.append({
        'structure': label, 'mutation': mut_map.get(label, ''),
        'site_residues': n_total,
        'charged': n_charged, 'aromatic': n_aromatic,
        'hydrophobic': n_hydrophobic, 'polar': n_polar, 'glycine': n_gly,
        'M790 (gatekeeper)': has_met790,
        'C797 (irreversible target)': has_cys797,
    })
    cmd.delete('all')

site_df = pd.DataFrame(site_data).set_index('structure')

# Composition plot
comp_cols = ['charged', 'aromatic', 'hydrophobic', 'polar', 'glycine']
fig, ax = plt.subplots(figsize=(12, 5))
site_df[comp_cols].plot(kind='bar', stacked=True, ax=ax,
                         color=['#E74C3C', '#9B59B6', '#3498DB', '#2ECC71', '#F1C40F'],
                         edgecolor='black', linewidth=0.3)
ax.set_ylabel('Binding Site Residues (within 5\u00c5)')
ax.set_title('EGFR Binding Site Composition: WT vs Mutant')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Key residue table
print('\nKey Resistance Residues:')
site_df[['mutation', 'site_residues', 'M790 (gatekeeper)', 'C797 (irreversible target)']]

In [ ]:
#@title 8-2. Structural Overlay in 3D {display-mode: "form"}

STRUCT_A = 'WT_Erlotinib'  #@param {type:"string"}
STRUCT_B = 'T790M_Neratinib'  #@param {type:"string"}

if STRUCT_A in pipes and STRUCT_B in pipes:
    pa, pb = pipes[STRUCT_A], pipes[STRUCT_B]

    v = py3Dmol.view(width=900, height=600)
    v.removeAllModels()

    # Structure A (blue)
    v.addModel(open(pa.prot_H).read(), 'pdb')
    v.getModel().setStyle({'cartoon': {'color': '#3498DB', 'opacity': 0.7}})

    # Ligand A (blue sticks)
    if os.path.exists(pa.lig_H):
        v.addModel(open(pa.lig_H).read(), 'mol2')
        v.getModel().setStyle({}, {'stick': {'colorscheme': 'blueCarbon', 'radius': 0.2}})

    # Structure B (red)
    v.addModel(open(pb.prot_H).read(), 'pdb')
    v.getModel().setStyle({'cartoon': {'color': '#E74C3C', 'opacity': 0.7}})

    # Ligand B (red sticks)
    if os.path.exists(pb.lig_H):
        v.addModel(open(pb.lig_H).read(), 'mol2')
        v.getModel().setStyle({}, {'stick': {'colorscheme': 'redCarbon', 'radius': 0.2}})

    v.zoomTo()
    print(f'Blue: {STRUCT_A} ({pa.mutation}), Red: {STRUCT_B} ({pb.mutation})')
    v
else:
    print(f'Available: {list(pipes.keys())}')

---
## 9. 3D Visualization

In [ ]:
#@title 9-1. View Any Structure + Docking Result {display-mode: "form"}

VIEW_STRUCT = 'WT_Erlotinib'  #@param {type:"string"}
VIEW_DRUG = 'native'  #@param {type:"string"}
POSE_IDX = 0  #@param {type:"integer"}

if VIEW_STRUCT in pipes:
    p = pipes[VIEW_STRUCT]
    avail = list(p.results.keys())
    if VIEW_DRUG in avail:
        print(f'{VIEW_STRUCT} ({p.mutation}) + {VIEW_DRUG}, pose #{POSE_IDX}')
        print(f'Magenta=reference, Green=docked pose')
        p.view_3d(VIEW_DRUG, POSE_IDX)
    else:
        print(f'Available dockings for {VIEW_STRUCT}: {avail}')
else:
    print(f'Available structures: {list(pipes.keys())}')

In [ ]:
#@title 9-2. Gallery: 2D Ligand Structures {display-mode: "form"}

gallery_mols = []
gallery_labels = []

for label, p in pipes.items():
    sdf = p.results.get('native')
    if sdf:
        suppl = Chem.SDMolSupplier(sdf, True)
        if len(suppl) > 0 and suppl[0] is not None:
            mol = Chem.RemoveHs(suppl[0])
            mol.RemoveAllConformers()
            AllChem.Compute2DCoords(mol)
            gallery_mols.append(mol)
            df = p.get_scores('native')
            best = df['minimizedAffinity'].min() if not df.empty else 0
            cx = next(c for c in EGFR_COMPLEXES if c['label'] == label)
            gallery_labels.append(f"{label}\n{cx['mutation']}\n{best:.1f} kcal/mol")

if gallery_mols:
    img = Draw.MolsToGridImage(gallery_mols, legends=gallery_labels,
                                molsPerRow=4, subImgSize=(350, 250))
    display(img)

---
## 10. Summary & Export

In [ ]:
#@title 10-1. Full Summary {display-mode: "form"}

full_summary = []
for cx in EGFR_COMPLEXES:
    row = {
        'Structure': cx['label'], 'PDB': cx['pdb'],
        'Mutation': cx['mutation'], 'Drug (co-crystal)': cx['drug'],
        'Generation': cx['generation'], 'Prepared': cx['label'] in pipes,
    }
    if cx['label'] in pipes:
        p = pipes[cx['label']]
        df = p.get_scores('native')
        if not df.empty:
            row['Re-dock Best'] = f"{df['minimizedAffinity'].min():.2f}"

        # Cross-docking scores
        for drug_name in REFERENCE_DRUGS:
            key = (drug_name, cx['label'])
            if key in xdock_scores:
                row[f'vs {drug_name}'] = f"{xdock_scores[key]:.2f}"

    full_summary.append(row)

summary_final = pd.DataFrame(full_summary)
summary_final

In [ ]:
#@title 10-2. Export Results {display-mode: "form"}
import shutil

# Summary CSV
summary_final.to_csv(f'{WORK_ROOT}/egfr_summary.csv', index=False)

# Cross-docking matrix
matrix.to_csv(f'{WORK_ROOT}/crossdock_matrix.csv')

# All cross-docking scores
xdock_df.to_csv(f'{WORK_ROOT}/crossdock_all_scores.csv', index=False)

# Interaction networks
for label, (fp, df, ligs) in prolif_data.items():
    try:
        net = LigNetwork.from_ifp(df, ligs[0], kind='frame', frame=0, rotation=270)
        net.save(f'{WORK_ROOT}/{label}/{label}_interactions.html')
    except Exception:
        pass

# Zip
zip_path = '/content/egfr_wt_mutant_results'
shutil.make_archive(zip_path, 'zip', WORK_ROOT)
print(f'Archive: {zip_path}.zip')

try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print(f'Results at: {WORK_ROOT}')

---
## Appendix: Clinical Interpretation Guide

### Expected Results

| Observation | Clinical Meaning |
|-------------|------------------|
| Erlotinib binds WT and L858R well, but weakly to T790M | T790M confers resistance to 1st gen |
| Afatinib binds broadly but modestly to T790M | 2nd gen partially overcomes T790M |
| Osimertinib binds T790M strongly, WT weakly | 3rd gen is T790M-selective |
| Lapatinib structure shows DFG-out pocket | Type II inhibitors access different pocket |

### Key Residues

| Residue | Role |
|---------|------|
| **T790** | Gatekeeper: T→M increases hydrophobic bulk, blocks 1st gen |
| **L858** | Activation loop: L→R stabilizes active conformation |
| **C797** | 3rd gen target: irreversible binding site for osimertinib |
| **L718** | Hinge region contact for most TKIs |
| **M793** | Hinge hydrogen bond acceptor |